# 02 EDA And Statistics

This notebook summarizes the exploratory and statistical findings. Each chart is tied to a question, observation, interpretation, and modeling implication.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

DB_PATH = Path('../data/processed/airfare.duckdb')
con = duckdb.connect(DB_PATH.as_posix(), read_only=True)

## Fare Distribution By Route

Question: How are fare levels distributed across the two selected routes?

Observation: LAX-JFK is generally more expensive than ATL-BOS, and both routes are right-skewed.

Implication: medians and quantiles are more robust than means alone, and error analysis should separate common fares from high-price tails.

![Total fare distribution](../docs/images/total_fare_distribution.png)

## Median Fare By Lead Time

Question: How does fare vary with days before departure?

Observation: ATL-BOS gets more expensive close to departure, while LAX-JFK is flatter and noisier.

Implication: route and lead time interact. A single global rule like "book 31-45 days early" would be too simplistic.

![Median fare by lead time](../docs/images/median_fare_by_lead_time.png)

## Repeated Itinerary Fare Path

Question: Do fares for the same itinerary move over repeated search dates?

Observation: individual fare paths can jump sharply.

Implication: BOOK/WAIT is measurable in this dataset, but individual fare movement is noisy and needs careful evaluation.

![Repeated leg fare path](../docs/images/repeated_leg_fare_path.png)

## SQL Example: Lead-Time Buckets

In [ ]:
con.execute('''
select
    route,
    case
        when lead_time_days between 1 and 7 then '01-07 days'
        when lead_time_days between 8 and 14 then '08-14 days'
        when lead_time_days between 15 and 30 then '15-30 days'
        when lead_time_days between 31 and 45 then '31-45 days'
        else '46-60 days'
    end as lead_time_bucket,
    count(*) as row_count,
    round(median(totalFare), 2) as median_total_fare
from airfare_clean
group by route, lead_time_bucket
order by route, lead_time_bucket
''').fetchdf()

## Statistical Question: Are Nonstop Flights More Expensive?

Hypothesis tests were run by route instead of globally. The result was route-dependent:

- ATL-BOS nonstop fares had about a $27 higher median fare.
- LAX-JFK nonstop fares had about a $9 lower median fare.

This supports a careful claim: stop-type relationships differ by route. It does not support a universal claim that nonstop flights are always more expensive.

In [ ]:
pd.read_csv('../reports/tables/phase08_stop_type_stat_tests.csv')